In [ ]:
import os
import glob
import numpy as np
import cv2 as cv
import torch
import torch.nn as nn
import torch.nn.functional as functional
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ntpath
import random
import math

def set_global_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_global_seed(42)

try:
    import torch_directml
    device = torch_directml.device()
    print(f"hardware, using DirectML ({device})")
except:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"hardware, using CUDA for Nvidia or CPU{device}")

class Config:
    base_directory = '../../../../'
    train_directory = os.path.join(base_directory, 'antrenare')
    valid_directory = os.path.join(base_directory, 'validare')
    output_directory = "output"
    os.makedirs(output_directory, exist_ok=True)

    character_mapping = {'fred': 1, 'daphne': 2, 'shaggy': 3, 'velma': 4}
    id_to_character = {1: 'fred', 2: 'daphne', 3: 'shaggy', 4: 'velma'}
    total_classes = 5

    model_filename = "model2.pth"
    patch_size = 36
    batch_size = 64
    epochs = 15
    learning_rate = 0.001
    max_mining_images = 300

In [ ]:
def augment_image(patch):
    variations = [patch]
    variations.append(cv.flip(patch, 1))

    height, width = patch.shape
    small = cv.resize(patch, (int(width * 0.7), int(height * 0.7)))
    variations.append(cv.resize(small, (width, height)))

    contrast = np.random.uniform(0.6, 1.4)
    brightness = np.random.randint(-40, 40)
    variations.append(cv.convertScaleAbs(patch, alpha=contrast, beta=brightness))

    return variations

def get_data(validation_split=0.15):
    all_image_paths = []
    for char in Config.character_mapping.keys():
        all_image_paths.extend(glob.glob(os.path.join(Config.train_directory, char, '*.jpg')))

    random.Random(42).shuffle(all_image_paths)
    split_index = int(len(all_image_paths) * (1 - validation_split))
    train_files = all_image_paths[:split_index]
    valid_files = all_image_paths[split_index:]

    def process_images(file_list, is_training=True):
        patches, labels = [], []

        for path in tqdm(file_list, leave=False):
            image = cv.imread(path, cv.IMREAD_GRAYSCALE)
            if image is None: continue

            char_folder = os.path.basename(os.path.dirname(path))
            annotation_file = os.path.join(Config.train_directory, f"{char_folder}_annotations.txt")
            all_annotations = np.loadtxt(annotation_file, dtype=str)
            if all_annotations.ndim == 1:
                all_annotations = np.array([all_annotations])

            image_annotations = all_annotations[all_annotations[:, 0] == ntpath.basename(path)]

            for row in image_annotations:
                x1, y1, x2, y2 = int(row[1]), int(row[2]), int(row[3]), int(row[4])
                char_name = row[5]

                if char_name not in Config.character_mapping: continue

                label_id = Config.character_mapping[char_name]
                face = cv.resize(image[y1:y2, x1:x2], (Config.patch_size, Config.patch_size))

                if is_training:
                    for augmented_patch in augment_image(face):
                        patches.append(augmented_patch / 255.0)
                        labels.append(label_id)
                else:
                    patches.append(face / 255.0)
                    labels.append(label_id)

            img_h, img_w = image.shape
            negative_count = 20 if is_training else 10

            for _ in range(negative_count):
                size = np.random.randint(40, 150)
                if img_w - size <= 0 or img_h - size <= 0: continue
                x = np.random.randint(0, img_w - size)
                y = np.random.randint(0, img_h - size)

                is_valid_negative = True
                for row in image_annotations:
                    face_box = [int(row[1]), int(row[2]), int(row[3]), int(row[4])]
                    newx1, newy1 = max(x, face_box[0]), max(y, face_box[1])
                    newx2, newy2 = min(x + size, face_box[2]), min(y + size, face_box[3])
                    if max(0, newx2 - newx1) * max(0, newy2 - newy1) > 0:
                        is_valid_negative = False
                        break

                if is_valid_negative:
                    negative_patch = cv.resize(image[y:y+size, x:x+size], (Config.patch_size, Config.patch_size))
                    patches.append(negative_patch / 255.0)
                    labels.append(0)

        return np.array(patches, dtype=np.float32), np.array(labels, dtype=np.longlong)

    train_X, train_y = process_images(train_files, True)
    valid_X, valid_y = process_images(valid_files, False)

    return (train_X, train_y), (valid_X, valid_y), train_files

(t_X, t_y), (v_X, v_y), train_file_list = get_data()

In [ ]:
class CNNforface2(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Conv2d(128, 256, 4, padding=0), nn.ReLU(), nn.Dropout(0.5),
            nn.Conv2d(256, Config.total_classes, 1)
        )
        self.to(device)

    def forward(self, x):
        return self.classifier(self.features(x))

def run_training(train_X, train_y, valid_X, valid_y):
    train_X = train_X.reshape(-1, 1, 36, 36)
    valid_X = valid_X.reshape(-1, 1, 36, 36)

    train_dataset = TensorDataset(torch.tensor(train_X), torch.tensor(train_y))
    valid_dataset = TensorDataset(torch.tensor(valid_X), torch.tensor(valid_y))

    train_loader = DataLoader(train_dataset, batch_size=Config.batch_size, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=Config.batch_size, shuffle=False)

    model = CNNforface2()
    optimizer = optim.Adam(model.parameters(), lr=Config.learning_rate)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(Config.epochs):
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            output = model(images).view(images.size(0), Config.total_classes)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if (epoch + 1) % 5 == 0:
            model.eval()
            correct, total = 0, 0
            with torch.no_grad():
                for images, labels in valid_loader:
                    images, labels = images.to(device), labels.to(device)
                    output = model(images).view(images.size(0), Config.total_classes)
                    predictions = torch.argmax(output, dim=1)
                    correct += (predictions == labels).sum().item()
                    total += labels.size(0)
            print(f"epoch {epoch+1}, loss: {total_loss/len(train_loader)}, validation Accuracy: {correct/total}")

    return model

model_path = os.path.join(Config.output_directory, Config.model_filename)

if os.path.exists(model_path):
    trained_model = CNNforface2()
    trained_model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    print(f"loaded existing model from {Config.model_filename}")
else:
    trained_model = run_training(t_X, t_y, v_X, v_y)
    torch.save(trained_model.state_dict(), model_path)
    print(f"model saved to {Config.model_filename}")

In [ ]:
Config.scale_factor = 0.85
Config.detection_stride = 8
mined_negatives_path = os.path.join(Config.output_directory, "mined_negatives.npy")

def detect_image(image_path, threshold, model):
    model.eval()
    image_gray = cv.imread(image_path, cv.IMREAD_GRAYSCALE)
    if image_gray is None: return {}

    detection_results = {character: [] for character in Config.character_mapping.keys()}

    current_image = image_gray.copy()
    current_scale = 1.0

    while current_image.shape[0] >= Config.patch_size and current_image.shape[1] >= Config.patch_size:
        image_tensor = torch.tensor(current_image.astype(np.float32) / 255.0).unsqueeze(0).unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(image_tensor)
            probabilities = functional.softmax(output, dim=1)
            confidence_values, class_indices = torch.max(probabilities, dim=1)

        for character_name, character_id in Config.character_mapping.items():
            rows, cols = torch.where((class_indices[0] == character_id) & (confidence_values[0] > threshold))

            if len(rows) > 0:
                rows, cols = rows.cpu().numpy(), cols.cpu().numpy()
                scores = confidence_values[0, rows, cols].cpu().numpy()
                inverse_scale = 1.0 / current_scale

                for r, c, score in zip(rows, cols, scores):
                    x = int(c * Config.detection_stride * inverse_scale)
                    y = int(r * Config.detection_stride * inverse_scale)
                    w = int(Config.patch_size * inverse_scale)
                    detection_results[character_name].append([x, y, x + w, y + w, float(score)])

        current_image = cv.resize(current_image, None, fx=Config.scale_factor, fy=Config.scale_factor)
        current_scale *= Config.scale_factor

    return detection_results

def negative_mining_func(model, train_files, train_X, train_y, valid_X, valid_y):
    final_model_path = os.path.join(Config.output_directory, Config.model_filename)

    if os.path.exists(mined_negatives_path):
        print("loading, we have previously mined hard negatives")
        hard_negatives = np.load(mined_negatives_path)
    else:
        hard_negatives = []
        subset = random.sample(train_files, min(len(train_files), Config.max_mining_images))

        for path in tqdm(subset, desc="starting mining"):
            results = detect_image(path, 0.6, model)
            
            all_detections = []
            for detections in results.values():
                all_detections.extend(detections)

            if not all_detections: continue

            char_folder = os.path.basename(os.path.dirname(path))
            annotation_file = os.path.join(Config.train_directory, f"{char_folder}_annotations.txt")
            all_annotations = np.loadtxt(annotation_file, dtype=str)
            if all_annotations.ndim == 1: all_annotations = np.array([all_annotations])
            
            image_annotations = all_annotations[all_annotations[:, 0] == ntpath.basename(path)]

            image_gray = cv.imread(path, cv.IMREAD_GRAYSCALE)
            for box in all_detections:
                x1, y1, x2, y2 = map(int, box[:4])
                is_false_positive = True

                for row in image_annotations:
                    face_box = [int(row[1]), int(row[2]), int(row[3]), int(row[4])]
                    ix1, iy1 = max(x1, face_box[0]), max(y1, face_box[1])
                    ix2, iy2 = min(x2, face_box[2]), min(y2, face_box[3])
                    if max(0, ix2 - ix1) * max(0, iy2 - iy1) > 0: 
                        is_false_positive = False
                        break

                if is_false_positive:
                    if y2 > image_gray.shape[0] or x2 > image_gray.shape[1] or x1 < 0 or y1 < 0: continue
                    crop = cv.resize(image_gray[y1:y2, x1:x2], (Config.patch_size, Config.patch_size))
                    hard_negatives.append(crop / 255.0)

        hard_negatives = np.array(hard_negatives, dtype=np.float32)
        np.save(mined_negatives_path, hard_negatives)
        print(f"mining done, found and saved {len(hard_negatives)} hard negatives")

    if len(hard_negatives) > 0:
        user_choice = input(f"found {len(hard_negatives)} negatives. do you want to retrain the multi-class model? (y/n): ").lower()
        if user_choice == 'y':
            print("retraining model with hard negatives")
            negative_labels = np.zeros(len(hard_negatives), dtype=np.longlong)

            new_train_X = np.concatenate([train_X, hard_negatives])
            new_train_y = np.concatenate([train_y, negative_labels])

            model = run_training(new_train_X, new_train_y, valid_X, valid_y)
            torch.save(model.state_dict(), final_model_path)
            print(f"model saved to {final_model_path}")
        else:
            print("skipping retraining as per user request.")
    else:
        print("no negatives found to retrain with")

    return model

trained_model = negative_mining_func(trained_model, train_file_list, t_X, t_y, v_X, v_y)

In [ ]:
test_directory = os.path.join(Config.base_directory, 'evaluare/fake_test')
test_images = sorted(glob.glob(os.path.join(test_directory, '*.jpg')))

def detect_sliding_window(image_path, threshold, model, stride=6):
    model.eval()
    image_gray = cv.imread(image_path, cv.IMREAD_GRAYSCALE)
    if image_gray is None: return {}

    results = {character: [] for character in Config.character_mapping.keys()}
    current_image = image_gray.copy()
    current_scale = 1.0

    while current_image.shape[0] >= Config.patch_size and current_image.shape[1] >= Config.patch_size:
        h, w = current_image.shape
        batch_patches, batch_coords = [], []

        for y in range(0, h - Config.patch_size, stride):
            for x in range(0, w - Config.patch_size, stride):
                patch = current_image[y : y + Config.patch_size, x : x + Config.patch_size]
                batch_patches.append(patch.astype(np.float32) / 255.0)
                batch_coords.append((x, y))

                if len(batch_patches) >= 64:
                    tensors = torch.tensor(np.array(batch_patches)).unsqueeze(1).to(device)
                    with torch.no_grad():
                        output = model(tensors).view(len(tensors), -1)
                        probs = torch.nn.functional.softmax(output, dim=1)
                        scores, class_ids = torch.max(probs, dim=1)

                    scores = scores.cpu().numpy()
                    class_ids = class_ids.cpu().numpy()

                    for i, (score, cls_id) in enumerate(zip(scores, class_ids)):
                        if cls_id > 0 and score > threshold:
                            if cls_id in Config.id_to_character:
                                char_name = Config.id_to_character[cls_id]
                                bx, by = batch_coords[i]
                                real_x, real_y = int(bx / current_scale), int(by / current_scale)
                                real_size = int(Config.patch_size / current_scale)
                                results[char_name].append([real_x, real_y, real_x + real_size, real_y + real_size, float(score)])

                    batch_patches, batch_coords = [], []

        current_image = cv.resize(current_image, None, fx=Config.scale_factor, fy=Config.scale_factor)
        current_scale *= Config.scale_factor

    return results

def get_overlap_metrics(box_a, box_b):
    xa, ya = max(box_a[0], box_b[0]), max(box_a[1], box_b[1])
    xb, yb = min(box_a[2], box_b[2]), min(box_a[3], box_b[3])
    intersection = max(0, xb - xa + 1) * max(0, yb - ya + 1)
    area_a = (box_a[2] - box_a[0] + 1) * (box_a[3] - box_a[1] + 1)
    area_b = (box_b[2] - box_b[0] + 1) * (box_b[3] - box_b[1] + 1)
    iou = intersection / float(area_a + area_b - intersection + 1e-6)
    iom = intersection / float(min(area_a, area_b) + 1e-6)
    return iou, iom

def run_nms(boxes, scores, iou_limit=0.1, iom_limit=0.5):
    if len(boxes) == 0: return [], []
    boxes, scores = np.array(boxes), np.array(scores)
    indices = np.argsort(scores)[::-1]
    keep = []
    while len(indices) > 0:
        current = indices[0]
        keep.append(current)
        remove = [0]
        for pos in range(1, len(indices)):
            next_idx = indices[pos]
            iou, iom = get_overlap_metrics(boxes[current], boxes[next_idx])
            if iou > iou_limit or iom > iom_limit:
                remove.append(pos)
        indices = np.delete(indices, remove)
    return boxes[keep], scores[keep]

print(f"starting task 2 inference on {len(test_images)} images from {test_directory}")

final_results = {char: {'boxes':[], 'scores':[], 'names':[]} for char in Config.character_mapping.keys()}
confidence_threshold = 0.50

for path in tqdm(test_images, desc="task 2 inference"):
    raw_results = detect_sliding_window(path, 0.01, trained_model)

    for char, detections in raw_results.items():
        if not detections: continue
        detections = np.array(detections)
        mask = detections[:, 4] > confidence_threshold

        filtered_boxes = detections[mask, :4]
        filtered_scores = detections[mask, 4]

        nms_boxes, nms_scores = run_nms(filtered_boxes, filtered_scores)

        if len(nms_boxes) > 0:
            final_results[char]['boxes'].extend(nms_boxes)
            final_results[char]['scores'].extend(nms_scores)
            final_results[char]['names'].extend([ntpath.basename(path)] * len(nms_scores))

for char in Config.character_mapping.keys():
    np.save(os.path.join(Config.output_directory, f"detections_{char}.npy"), np.array(final_results[char]['boxes']))
    np.save(os.path.join(Config.output_directory, f"scores_{char}.npy"), np.array(final_results[char]['scores']))
    np.save(os.path.join(Config.output_directory, f"file_names_{char}.npy"), np.array(final_results[char]['names']))

print(f"Inference complete. Files saved to {Config.output_directory}")

character_colors = {'fred': 'blue', 'daphne': 'yellow', 'shaggy': 'green', 'velma': 'purple'}
plt.figure(figsize=(20, 40))
num_to_show = min(len(test_images), 50)
cols = 5
rows = math.ceil(num_to_show / cols)

for i, path in enumerate(test_images[:num_to_show]):
    filename = ntpath.basename(path)
    image = cv.imread(path)
    if image is None: continue
    image = cv.cvtColor(image, cv.COLOR_BGR2RGB)
    ax = plt.subplot(rows, cols, i+1)
    ax.imshow(image)
    ax.axis('off')
    ax.set_title(filename, fontsize=9)

    for char in Config.character_mapping.keys():
        names = np.array(final_results[char]['names'])
        dets = np.array(final_results[char]['boxes'])
        scrs = np.array(final_results[char]['scores'])

        if len(names) == 0: continue
        mask = (names == filename)
        for b, s in zip(dets[mask], scrs[mask]):
            ax.add_patch(patches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1],
                                         linewidth=2, edgecolor=character_colors[char], facecolor='none'))
            ax.text(b[0], b[1]-5, f"{char[0].upper()} {s:.2f}",
                    color=character_colors[char], fontsize=8, weight='bold')

plt.tight_layout()
plt.show()